# <DEVICE> — Measure

`metadata.hera.policy` — scientific record: append new cells; never edit or
delete executed ones. (Who *executes* a measure cell is set by the session
mode — see the brief.)

Confirm each `# <-- verify` (rig: addresses) and `# <-- validate` (device:
operating limits, wiring) line at bring-up. Cells marked **PATTERN** /
**PROPOSAL** carry `<PLACEHOLDER>` values — review before running.

## Setup

In [ ]:
import os
import time

import numpy as np
import scipy.constants as cst

import qcodes
from qcodes import validators
from qcodes.instrument_drivers.stanford_research import SR830, SR860
from qcodes.instrument_drivers.tektronix.Keithley_2400 import Keithley_2400

import sweep
from sweep import interruptible_sleep
from gui.qt_progress import QtProgress
sweep.set_progress(QtProgress())

from hera.analysis import GateCalibration
from sharpelab_nb.drivers import SRS_DC205

In [ ]:
main_dir = os.getcwd()
sweep.set_basedir(os.path.join(main_dir, "data"))
main_dir

# Connect Instruments

## Thermometry — Leiden DR TempControl (ActiveX/COM)

Temperature is read from the DR TempControl (TC) app over COM — **the TC app
must be running**; there is no file-based fallback for the kernel (rig card).
A closed or frozen TC app makes reads stale or raise — re-run the handle cell
to reconnect. Values are **mK**.

In [ ]:
import win32com.client

class LVApp:

    def __init__(self, AppName, ViName):
        self.App = win32com.client.Dispatch(AppName)
        self.Vi = self.App.GetViReference(ViName)

    # Reading data from VI's is straight forward
    def GetData(self, ControlName):
        return self.Vi.GetControlValue(ControlName)

    # Writing data to VI's is more complicated.
    # It is possible to write data by simply using SetControlValue(<Name>), but that does not trigger ValueChanged event
    # in LabView program. Since a lot of things are happening inside the event structure, it breaks TC and FP programs.
    # Instead there is a special cluster called "SetControl". It has three fields: "Name", "Scalar" and "Array".
    # To write data to any control put it's name in "Name" field and value in either "Scalar" or "Array" field depending on
    # data type. Do not write data in both "Scalar" and "Array" fields as the LabView program will ignore the command. Put
    # empty string '' or empty list [[], []] in the unused field.
    # It is possible to write data to individual controls in clusters by <ClusterName>.<ControlName> notation. It will
    # trigger the ValueChanged event for that control and not the cluster.
    # After the program reads the "SetControl" structure it will empty it to flag that it's been processed. Checking if
    # the cluster is empty allows synchronous operation
    def SetData(self, ControlName, ControlData, Async = False):
        if type(ControlData) in (tuple, list):
            self.Vi.SetControlValue('SetControl', (ControlName, '', ControlData))
        else:
            self.Vi.SetControlValue('SetControl', (ControlName, ControlData, [[], []]))
        if not Async:
            while self.Vi.GetControlValue('SetControl')[0] != '': time.sleep(0.1)


# For compiled executables:
TC = LVApp("DRTempControl.Application",
           r"C:\Program Files (x86)\Leiden Cryogenics\TC\DR TempControl.exe\TC.vi")

In [ ]:
# Stage thermometers per the rig-card channel map (values in mK).
# The sample sits on the PROBE mixing chamber: probe_mc_temp is the
# sample-stage thermometer.
fridge_3k_temp = qcodes.Parameter('fridge_3k_temp', unit='mK', get_cmd=lambda: TC.GetData('T0'))
probe_3k_temp = qcodes.Parameter('probe_3k_temp', unit='mK', get_cmd=lambda: TC.GetData('T7'))
fridge_still_temp = qcodes.Parameter('fridge_still_temp', unit='mK', get_cmd=lambda: TC.GetData('T1'))
probe_still_temp = qcodes.Parameter('probe_still_temp', unit='mK', get_cmd=lambda: TC.GetData('T6'))
fridge_50mK_temp = qcodes.Parameter('fridge_50mK_temp', unit='mK', get_cmd=lambda: TC.GetData('T4'))
probe_50mK_temp = qcodes.Parameter('probe_50mK_temp', unit='mK', get_cmd=lambda: TC.GetData('T3'))
fridge_mc_temp = qcodes.Parameter('fridge_mc_temp', unit='mK', get_cmd=lambda: TC.GetData('T2'))
probe_mc_temp = qcodes.Parameter('probe_mc_temp', unit='mK', get_cmd=lambda: TC.GetData('T5'))

for _t in (fridge_3k_temp, probe_3k_temp, fridge_still_temp, probe_still_temp,
           fridge_50mK_temp, probe_50mK_temp, fridge_mc_temp, probe_mc_temp):
    print(f"  {_t.name}: {_t()} mK")

## Probe-MC temperature control (optional)

Closed-loop control of the probe mixing chamber, through the same TC handle.
Run only when a temperature ladder is planned — it turns the heater loop on
(rig state). Setpoints are **mK**; settle is dwell-based, no PID readback
(rig card).

In [ ]:
# PATTERN — probe-MC closed-loop control. Review before running; turn the
# loop off after the ladder (TC.SetData("Mode1 ON", False)).
TC.SetData("CS Mode", 1)

def set_probe_temp(temp_mK):
    TC.SetData("I_list", [[0, 0, 0, temp_mK]])

TC.SetData("Mode1 ON", True)

# Rebind the sample thermometer as settable so ladders sweep it directly.
probe_mc_temp = qcodes.Parameter('probe_mc_temp', unit='mK',
                                 get_cmd=lambda: TC.GetData('T5'),
                                 set_cmd=set_probe_temp)

## Lock-ins

In [ ]:
# Connect what's wired this cooldown; comment out the rest.
sr0 = SR860('sr0', "TCPIP0::192.168.0.111")  # <-- verify
sr1 = SR860('sr1', "TCPIP0::192.168.0.112")  # <-- verify
sr2 = SR860('sr2', "TCPIP0::192.168.0.113")  # <-- verify
sr3 = SR830('sr3', "GPIB1::4::INSTR")  # <-- verify
sr4 = SR830('sr4', "GPIB1::5::INSTR")  # <-- verify
sr5 = SR830('sr5', "GPIB0::9::INSTR")  # <-- verify

# Warm the qcodes cache for time_constant + filter_slope so they land in
# the run-metadata instruments snapshot (the snapshot serializes cached
# values; without a read they come back None and tau is unrecoverable from
# disk). The swept-point dwell floor is k*tau (STYLE rule 12), so record
# tau at bring-up.
for _sr in (sr0, sr1, sr2, sr3, sr4, sr5):
    print(f"  {_sr.name}: tau={_sr.time_constant()} s, slope={_sr.filter_slope()} dB/oct")

## Back gate (Keithley 2400)

In [ ]:
k_bg = Keithley_2400('k_bg', 'GPIB0::2::INSTR')  # <-- verify address — back gate
k_bg.mode.set('VOLT')
k_bg.volt.set(0)  # park at 0 (bring-up happens at rest)
k_bg.rangei.set(1e-6)
k_bg.rangev.set(21)
k_bg.compliancev.set(20)
k_bg.compliancei.set(1e-6)
k_bg.volt.step = 0.005
k_bg.nplci.set(10)

## DC bias (SRS DC205)

In [ ]:
# sharpelab_nb driver: ramp_voltage = 100-point linear ramp + mandatory 10 s
# settle when |dV| > 0.1 V (same semantics as the rig's historical driver).
# smart_voltage is the ramped settable parameter (data column
# 'dcs_bias_smart_voltage'); direct .voltage sets skip the ramp — avoid.
# Range codes: 0 = ±1 V, 1 = ±10 V, 2 = ±100 V.
dcs_bias = SRS_DC205("dcs_bias", "ASRL3::INSTR")  # <-- verify address — DC bias
print(f"range code: {dcs_bias.voltage_range()}  V = {dcs_bias.voltage():+.6f}")

## Magnet (AMI430)

Read-only monitoring is the launcher's magnet worker; the kernel connects
only for field sessions. Field range, coil constant, and interrupt semantics
are unestablished (rig card — Gaps): confirm with the czar before ramping.

In [ ]:
# Connect only for a field session; comment out otherwise.
from qcodes.instrument_drivers.american_magnetics import AMIModel430

mag = AMIModel430("mag", address="TCPIP0::192.168.0.185::7180::SOCKET")  # <-- verify
mag.ramp_rate(0.0005)  # T/s — operator practice (rig card)    # <-- validate
print(f"B = {mag.field():+.4f} T   ramp_rate = {mag.ramp_rate()} T/s")

# SAFETIES

In [ ]:
# slot: gate-validators — operating limits from the device card's gate-
# operating-limits table; commit EXACTLY the card numbers. Each value
# `# <-- validate`. Fences go on the RAW instrument parameters.
BG_MIN, BG_MAX = <BG_MIN>, <BG_MAX>          # back gate (k_bg), V           # <-- validate
BIAS_MIN, BIAS_MAX = <BIAS_MIN>, <BIAS_MAX>  # DC bias (dcs_bias), V         # <-- validate
k_bg.volt.vals = validators.Numbers(BG_MIN, BG_MAX)
dcs_bias.smart_voltage.vals = validators.Numbers(BIAS_MIN, BIAS_MAX)
dcs_bias.voltage.vals = validators.Numbers(BIAS_MIN, BIAS_MAX)  # ramp_voltage sets via .voltage — fences ramps too

bg_lims = [BG_MIN, BG_MAX]

# Temperature setpoint cap — TYPO GUARD ONLY (set_probe_temp(3000) vs 300 mK).
# Only meaningful if the probe-MC control block was run.
T_SESSION_CAP_MK = <T_CAP_MK>  # mK    # <-- validate
probe_mc_temp.vals = validators.Numbers(0, T_SESSION_CAP_MK)

# Session field cap — machine-enforced: rejects any mag.field(target) with
# |B| > B_SESSION_CAP before hardware traffic. Only meaningful if the magnet
# was connected above.
B_SESSION_CAP = <B_CAP>  # Tesla    # <-- validate (session cap from card/brief)
mag.field.vals = validators.Numbers(-B_SESSION_CAP, B_SESSION_CAP)

# Helpers

In [ ]:
def spoints(a, b, c):
    return np.linspace(a, b, c)


def calc_time(vec1, vec2, delay):
    """Pre-flight sweep cost: seconds for a vec1 x vec2 raster at *delay* s/pt."""
    return len(vec1) * len(vec2) * delay

In [ ]:
# slot: geometry — gate-stack thicknesses from the device card's Gate-stack
# geometry (geometry prior — calibrate cnp-and-dline-calibration before
# trusting derived scales). Each value `# <-- validate`.
t_tg = <T_TG>      # top-gate hBN thickness, m    # <-- validate
t_bg = <T_BG>      # back-gate hBN thickness, m    # <-- validate
t_bgSiO2 = <T_BGSIO2>  # SiO2 term, m — zero for a metal/graphite back gate    # <-- validate

# Session gate calibration. cal.n_D(V_tg, V_bg) -> (n [m^-2], D [V/m]);
# set n_offset / D_offset from cnp-and-dline-calibration once live.
cal = GateCalibration.from_thicknesses(t_tg, t_bg, t_bgSiO2)

# Measurement config

In [ ]:
# slot: meas-config — sr0_gain + channel/contact assignments from the device
# card's Measurement setup (current channel + gain, provided contact
# configuration). Each value `# <-- validate`. Always set 'comments' per run —
# structured, current, no inherited text. Lock-in state (amplitude, freq,
# sens, tau) is NOT recorded here — sweep snapshots followed instruments
# live at each run start; bias_channel tells hera-watch where the drive is.
meas_config = {
    "source": "<CONTACT>",  # drive injection contact    # <-- validate
    "sr0": "drain current on <CONTACT>",  # <-- validate
    "sr0_gain": <SR0_GAIN>,  # current preamp gain    # <-- validate
    "sr1": "<PAIR>",  # <-- validate
    "sr2": "<PAIR>",  # <-- validate
    "bias_resistor": <BIAS_RESISTOR>,  # AC drive series resistor, e.g. 2e8    # <-- validate
    "dc_bias_resistor": <DC_BIAS_RESISTOR>,  # DC205 series resistor, e.g. 100e6    # <-- validate
    "bias_channel": "sr0",  # drive lock-in; I_bias = its live amplitude / bias_resistor (auto-snapshotted)    # <-- validate
    "sample_temperature_mK": probe_mc_temp(),  # TC-COM read (mK) — re-run this cell to refresh
    "comments": "<SET PER RUN>",
}